# iSeg-2017 — frugal 2.5D U-Net

CSF / grey matter / white matter segmentation on T1-T2 MRI of 6-month-old infants.

This notebook only **orchestrates** calls into the `iseg/` package, versioned in git.

**Before running**: Runtime → Change runtime type → **T4 GPU**.

## 1. Code, cache and dependencies

Upload `iseg-colab.zip` (code + preprocessed cache, 26 MB). The cache saves transferring
the 361 MB of raw MRI — `build_cache` skips subjects already cached without ever opening
the `.hdr/.img`.

In [ ]:
from google.colab import files
files.upload()          # pick iseg-colab.zip

!mkdir -p /content/repo && unzip -q -o iseg-colab.zip -d /content/repo
%cd /content/repo
!pip -q install nibabel onnx onnxruntime onnxscript

import torch, os
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to T4')
print('cache:', len(os.listdir('cache')), 'subjects')

## 2. Training

8 subjects to learn, subjects 1 and 2 held out to measure. The 3D Dice on those two
prints every 5 epochs; the weights of the best pass are written to `runs/<variant>.pt`.

Around 10 min on a T4.

In [ ]:
!python -m iseg.train --variant separable --epochs 60

The other two variants, if you want to compare:

| variant | params | int8 | Dice |
|---|---|---|---|
| `standard` | 1,943,636 | 1.89 MB | 0.8927 |
| `separable` | 386,782 | 0.43 MB | 0.8624 |
| `tiny` | 26,974 | 0.07 MB | 0.8281 |

In [ ]:
# !python -m iseg.train --variant standard --epochs 60
# !python -m iseg.train --variant tiny --epochs 60

## 3. Export and deploy into the web page

Produces the float32 `.onnx` and its int8 version (~3.5x smaller), and copies the latter
into `webdemo/` as `model-<variant>.onnx`, which is the name the page loads.

In [ ]:
!python -m iseg.export --checkpoint runs/separable.pt --deploy webdemo

## 4. Retrieve the site

`webdemo/` is a static folder: publish it as-is on GitHub Pages, Vercel or Netlify.

In [ ]:
!zip -qr webdemo.zip webdemo
from google.colab import files
files.download('webdemo.zip')